In [ ]:
from pathlib import Path

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import GenericMaterials, GenericStocks, MaterialIntensities
from imagematerials.preprocessing import get_preprocessing_data


In [ ]:
circularity_scenarios = ["base", "narrow","slow"]
climate_policy_scenario_dir = Path("..", "data", "raw", "IMAGE_CircoMod", "SSP2_narrow_slow_close") 
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

circular_economy_scenario_dirs = {
    scenario: scenario_base_path / scenario
    for scenario in circularity_scenarios
}


vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), climate_policy_scenario_dir, circular_economy_scenario_dirs)
bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), climate_policy_scenario_dir, circular_economy_scenario_dirs) 

In [ ]:
sectors = [vhc_sector, bld_sector]
ns_coordinates = {
    "Region": sectors[0].coordinates["Region"],
    "material": list(set(sectors[0].coordinates["material"] + sectors[1].coordinates["material"]))
}
ns_combined = Sector("combi", {}, coordinates=ns_coordinates)
sectors.append(ns_combined)

In [ ]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [ ]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")

@prism.interface
class SumMaterials(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("inflow_materials", )
    output_data: tuple[str] = ("summed_inflow_materials", )



    summed_inflow_materials: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Time, inflow_materials):
        t, dt = time.t, time.dt
        self.summed_inflow_materials[t].loc[:] = 0
        for inflow in inflow_materials:
            self.summed_inflow_materials[t].loc[{"material": inflow[t].coords["material"]}] += inflow[t].sum("Type")

In [ ]:
factory = ModelFactory(
    sectors, complete_timeline
    ).add(GenericStocks, ["buildings", "vehicles"]
    ).add(GenericMaterials,  "vehicles"
    ).add(MaterialIntensities, "buildings",
    ).add(SumMaterials, "combi", input_sources={"inflow_materials": ["vehicles", "buildings"]}
    )
model = factory.finish()

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

In [ ]:
model.combi["summed_inflow_materials"][2020]